In [ ]:
import os
import sys
# add the 'src' directory as one where we can import modules
PROJ_ROOT = os.path.join(os.pardir)
src_dir = os.path.join(PROJ_ROOT, 'src')
sys.path.append(src_dir)

print(os.getcwd())
print(sys.path)

import IPython.extensions.autoreload
%load_ext autoreload
%autoreload 2

# import my method from the source code
from ingest import data_workflow
from dataset import analyze, detect_elapsed_time_anomalies, resample
import matplotlib.pyplot as plt

In [ ]:
y = data_workflow("inariz")

# include the unit into the main column name
y.rename({"Valeur": "steam_consumption_(m3/h)"})

# for other columns, keep only numerical columns
# set the timestamp as a column with a standard name, not as the index
y = y.drop(["Unité"])


analyze(y)
elapsed_anomalies, expected_delta = detect_elapsed_time_anomalies(y, timestamp_col="measured_at")
# target
y = y[["measured_at", "steam_consumption_(m3/h)"]] #todo check, useless je pense (et faux ?)
y_10min = resample(y, desired_timedelta="10min", aggregate_function="mean")
y_15min = resample(y, desired_timedelta="15min", aggregate_function="mean")

Verify that the average is done correctly between [00:00; 00:15[

In [ ]:
import datetime 
y_15min[(y_15min["measured_at"] >= datetime.datetime(2025, 12, 15, 0, 0, 0)) & (y_15min["measured_at"] <= datetime.datetime(2025, 12, 15, 1, 0, 0))]

In [ ]:
y[(y["measured_at"] >= datetime.datetime(2025, 12, 15, 0, 15, 0)) & (y["measured_at"] < datetime.datetime(2025, 12, 15, 0, 30, 0))]["steam_consumption_(m3/h)"].describe()

In [ ]:
# plot original and resampled timeseries, to check the choice of aggregate_function
import PyQt6
%matplotlib qt
fig, ax = plt.subplots()

ax.plot(y["measured_at"], y["steam_consumption_(m3/h)"], label="original")
ax.step(y_10min["measured_at"], y_10min["steam_consumption_(m3/h)"], label="resampled 10 min", where="post")
ax.step(y_15min["measured_at"], y_15min["steam_consumption_(m3/h)"], label="resampled 15 min", where="post")
ax.set_ylabel("Value")
# ax.set_title(path.name)
ax.legend(loc="best")
plt.tight_layout()
plt.show()